In [45]:
import os
import docker
import tempfile
import shutil

from langgraph.prebuilt import create_react_agent
from dotenv import load_dotenv
from docker.errors import NotFound, ContainerError, APIError, ImageNotFound
load_dotenv()

True

In [ ]:
### Tools

In [ ]:
# Source Code Loader
def load_source_directory(dir_path: str) -> str:
    """
    Recursively loads a directory’s source code files into a JSON-compatible dictionary structure.

    Args:
        dir_path (str): Path to the directory to load.

    Returns:
        str: JSON string representing the directory structure and file contents.
    """
    def _dir_to_src(path: str) -> dict:
        structure = {}

        for item in os.listdir(path):
            full_path = os.path.join(path, item)
            rel_path = os.path.relpath(full_path, dir_path)

            if os.path.isdir(full_path):
                structure[item] = _dir_to_src(full_path)
            elif os.path.isfile(full_path):
                try:
                    with open(full_path, "r", encoding="utf-8") as f:
                        structure[item] = {
                            "path": rel_path,
                            "directory": os.path.dirname(full_path),
                            "extension": os.path.splitext(rel_path)[1],
                            "content": f.read()
                        }
                except Exception as e:
                    structure[item] = {
                        "path": rel_path,
                        "directory": os.path.dirname(full_path),
                        "extension": os.path.splitext(rel_path)[1],
                        "error": f"<Error reading file: {str(e)}>"
                    }
        return structure

    if not os.path.exists(dir_path):
        return json.dumps({"error": f"Directory not found: {dir_path}"}, indent=2)

    try:
        result = _dir_to_src(dir_path)
        return json.dumps(result, indent=2)
    except Exception as e:
        return json.dumps({"error": str(e)}, indent=2)

In [46]:
# Build Docker Image
def build_docker_image_from_string(
    dockerfile_content: str,
    context_path: str,
    image_tag: str
) -> str:
    """
    Builds a Docker image locally from a Dockerfile provided as a string,
    using the specified build context (relative or absolute path).

    The Dockerfile and temp folder are explicitly removed after the build.

    Args:
        dockerfile_content (str): Full Dockerfile content as a string.
        context_path (str): Path to use as Docker build context.
        image_tag (str): Tag name for the resulting Docker image.

    Returns:
        str: Success message with image ID or error message.
    """
    context_path = os.path.abspath(context_path)
    if not os.path.isdir(context_path):
        return f"❌ Error: context_path '{context_path}' is not a valid directory."

    client = docker.from_env()

    temp_dir = None

    try:
        # Create temp folder inside context_path (or system temp)
        temp_dir = tempfile.mkdtemp(prefix="docker_temp_")
        temp_dockerfile_path = os.path.join(temp_dir, "Dockerfile")

        # Write Dockerfile
        with open(temp_dockerfile_path, "w", encoding="utf-8") as f:
            f.write(dockerfile_content.replace("\r\n", "\n"))

        print(f"🔧 Building image '{image_tag}' using temporary Dockerfile...")
        image, logs = client.images.build(
            path=context_path,
            dockerfile=temp_dockerfile_path,
            tag=image_tag,
            rm=True
        )

        # Stream logs
        for chunk in logs:
            if "stream" in chunk:
                print(chunk["stream"].strip())
            elif "errorDetail" in chunk:
                print("❌ Build error:", chunk["errorDetail"].get("message", "Unknown error"))

        print(f"✅ Successfully built image '{image_tag}' (ID: {image.short_id})")
        return f"Successfully built image '{image_tag}' (ID: {image.short_id})"

    except docker.errors.BuildError as err:
        return f"❌ Build failed: {err}"
    except docker.errors.APIError as api_err:
        return f"❌ Docker API error: {api_err}"
    except Exception as e:
        return f"❌ Unexpected error: {e}"

    finally:
        client.close()

        # Explicitly delete temp Dockerfile and folder
        if temp_dir and os.path.exists(temp_dir):
            try:
                shutil.rmtree(temp_dir)
                print(f"🧹 Cleaned up temporary Dockerfile and folder: {temp_dir}")
            except Exception as cleanup_err:
                print(f"⚠️ Warning: Failed to clean temporary folder: {cleanup_err}")


In [ ]:
# Run Docker Image
def run_docker_image(
    image_tag: str,
    command: list = None,
    env_vars: dict = None,
    ports: dict = None,
    detach: bool = False
) -> dict:
    """
    Run a Docker image locally using the provided image_tag.

    Args:
        image_tag (str): The tag of the Docker image to run.
        command (list, optional): Command to run in the container, e.g., ["python", "app.py"].
        env_vars (dict, optional): Environment variables to pass to the container.
        ports (dict, optional): Port mappings, e.g., {"5000/tcp": 5000}.
        detach (bool, optional): Whether to run container in detached mode.

    Returns:
        dict: {
            "container_id": str,
            "message": str,
            "logs": Optional[str]
        }
    """
    client = docker.from_env()

    try:
        container = client.containers.run(
            image=image_tag,
            command=command,
            environment=env_vars,
            ports=ports,
            detach=detach,
            remove=not detach
        )

        if detach:
            return {
                "container_id": container.id,
                "message": f"✅ Container started in detached mode: {container.id[:12]}",
                "logs": None
            }

        # Stream logs and return them after completion
        logs_output = ""
        for log in container.logs(stream=True):
            decoded = log.decode("utf-8").strip()
            print(decoded)
            logs_output += decoded + "\n"

        container_id = getattr(container, "id", "unknown")
        return {
            "container_id": container_id,
            "message": "✅ Container ran successfully.",
            "logs": logs_output.strip()
        }

    except ImageNotFound:
        return {"container_id": None, "message": f"❌ Docker image '{image_tag}' not found locally.", "logs": None}
    except ContainerError as e:
        return {"container_id": None, "message": f"❌ Container exited with error: {str(e)}", "logs": None}
    except APIError as e:
        return {"container_id": None, "message": f"❌ Docker API error: {str(e)}", "logs": None}
    except Exception as e:
        return {"container_id": None, "message": f"❌ Unexpected error: {str(e)}", "logs": None}
    finally:
        client.close()

In [ ]:
# Test if Container is up and Running
def check_container_health(container_id: str) -> dict:
    """
    Check if a container is up and running healthy.

    Args:
        container_id (str): The ID or name of the container.

    Returns:
        dict: {
            "status": str,   # 'running', 'exited', etc.
            "health": str,   # 'healthy', 'unhealthy', 'none'
            "message": str   # diagnostic info
        }
    """
    client = docker.from_env()

    try:
        container = client.containers.get(container_id)
        container_state = container.attrs.get("State", {})
        status = container_state.get("Status", "unknown")
        health = container_state.get("Health", {}).get("Status", "none")

        if status == "running":
            if health == "healthy":
                msg = "Container is running and healthy."
            elif health == "unhealthy":
                msg = "Container is running but health check failed."
            else:
                msg = "Container is running (no health check defined)."
        else:
            msg = f"Container is not running (status: {status})."

        return {
            "status": status,
            "health": health,
            "message": msg
        }

    except NotFound:
        return {"status": "not_found", "health": "none", "message": "Container not found."}
    except APIError as e:
        return {"status": "error", "health": "none", "message": f"Docker API error: {e.explanation}"}
    except Exception as e:
        return {"status": "error", "health": "none", "message": str(e)}
    finally:
        client.close()

In [11]:
import json
import textwrap

def print_section(title: str, char: str = "=", width: int = 70):
    """Simple header separator."""
    print(f"\n{char * width}")
    print(f"{title.center(width)}")
    print(f"{char * width}\n")


def pretty_print_dict(d):
    """Pretty-print dicts or JSON strings."""
    if isinstance(d, str):
        try:
            d = json.loads(d)
        except json.JSONDecodeError:
            print(textwrap.indent(d.strip(), "  "))
            return
    print(json.dumps(d, indent=2))

In [53]:
agent = create_react_agent(
    model="gpt-5-nano",
    tools=[load_source_directory, build_docker_image_from_string, run_docker_image, check_container_health],
    prompt="""You are an expert DevOps engineer skilled in Docker and source code management. Your goal is to use provided tools
        in order to build docker image out of provided source code, run it and make sure everything is working as expected.
    """
)

C:\Users\tvaro\AppData\Local\Temp\ipykernel_19056\3403204451.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [70]:
def get_user_message(app_dir: str) -> dict:
    return {
        "role": "user",
        "content": (
            f"You are given a directory named '{app_dir}' in the current working directory. Please make everything necessary in order to have this application "
            "running inside a Docker container."
            "Extra Instructions: \n"
            "1. If you get an source code error you cant fix, terminate.\n"
            "2. Do not spam multiple containers."
            "3. If requirements are not provided do not try to change the code rather install necessary packages"
            "4. Do not include any files from current directory except the given one"
        )
    }

In [65]:
def stream_agent(user_message: list):
    print_section("📡 STARTING AGENT STREAM", "=")

    stats = {
        "agent_steps": 0,
        "tool_calls": 0,
        "messages": 0,
        "tokens": 0
    }

    for event in agent.stream({"messages": user_message}, stream_mode="updates"):
        for step, data in event.items():
            stats["agent_steps"] += 1
            print_section(f"🧩 STEP: {step}", "-")

            # handle message content
            for block in data["messages"][-1].content_blocks:
                stats["messages"] += 1
                block_type = block.get("type")

                if block_type == "tool_call":
                    stats["tool_calls"] += 1
                    print("🛠️  TOOL CALL:", block["name"])
                    print("   ARGS:")
                    pretty_print_dict(block["args"])

                elif block_type == "text":
                    text_content = block.get("text", "").strip()
                    try:
                        parsed = json.loads(text_content)
                        print("📦 TOOL RESULT:")
                        print(json.dumps(parsed, indent=2))
                    except json.JSONDecodeError:
                        print("💬 AGENT OUTPUT:")
                        print(textwrap.indent(textwrap.fill(text_content, width=80), "  "))

            # Try to collect token info if present
            token_info = data.get("metadata", {}).get("token_usage")
            if token_info:
                stats["tokens"] += token_info.get("total", 0)

    # --- print summary at the end ---
    print_section("📊 RUN SUMMARY", "=")
    print(f"🧩 Agent Steps:   {stats['agent_steps']}")
    print(f"🛠️ Tool Calls:    {stats['tool_calls']}")
    print(f"💬 Messages:      {stats['messages']}")
    if stats["tokens"]:
        print(f"🔢 Tokens Used:   {stats['tokens']}")
    print("=" * 70)

### APP

In [66]:
stream_agent([get_user_message("app")])


                       📡 STARTING AGENT STREAM                        


----------------------------------------------------------------------
                            🧩 STEP: agent                             
----------------------------------------------------------------------

🛠️  TOOL CALL: load_source_directory
   ARGS:
{
  "dir_path": "app"
}

----------------------------------------------------------------------
                            🧩 STEP: tools                             
----------------------------------------------------------------------

📦 TOOL RESULT:
{
  ".env": {
    "path": ".env",
    "directory": "app",
    "extension": "",
    "content": ""
  },
  "requirements.txt": {
    "path": "requirements.txt",
    "directory": "app",
    "extension": ".txt",
    "content": "mcp[cli]>=0.1.0\nhttpx>=0.24.0"
  },
  "server.py": {
    "path": "server.py",
    "directory": "app",
    "extension": ".py",
    "content": "# server.py -- minimal FastMCP server\nfrom mc

### APP No Req.txt

In [71]:
stream_agent([get_user_message("app-no-req")])


                       📡 STARTING AGENT STREAM                        


----------------------------------------------------------------------
                            🧩 STEP: agent                             
----------------------------------------------------------------------

🛠️  TOOL CALL: load_source_directory
   ARGS:
{
  "dir_path": "app-no-req"
}

----------------------------------------------------------------------
                            🧩 STEP: tools                             
----------------------------------------------------------------------

📦 TOOL RESULT:
{
  ".env": {
    "path": ".env",
    "directory": "app-no-req",
    "extension": "",
    "content": ""
  },
  "server.py": {
    "path": "server.py",
    "directory": "app-no-req",
    "extension": ".py",
    "content": "# server.py -- minimal FastMCP server\nfrom mcp.server.fastmcp import FastMCP\n\nmcp = FastMCP(\"MinimalDemo\", port=8000)\n\n# a simple tool the model can call\n@mcp.tool()\ndef add(

### APP Secrets

In [72]:
stream_agent([get_user_message("app-secrets")])


                       📡 STARTING AGENT STREAM                        


----------------------------------------------------------------------
                            🧩 STEP: agent                             
----------------------------------------------------------------------

🛠️  TOOL CALL: load_source_directory
   ARGS:
{
  "dir_path": "app-secrets"
}

----------------------------------------------------------------------
                            🧩 STEP: tools                             
----------------------------------------------------------------------

📦 TOOL RESULT:
{
  ".env": {
    "path": ".env",
    "directory": "app-secrets",
    "extension": "",
    "content": "MY_SECRET=pssst"
  },
  "requirements.txt": {
    "path": "requirements.txt",
    "directory": "app-secrets",
    "extension": ".txt",
    "content": "mcp[cli]>=0.1.0\nhttpx>=0.24.0"
  },
  "server.py": {
    "path": "server.py",
    "directory": "app-secrets",
    "extension": ".py",
    "content": 

In [ ]:
# TODO::
# Better Response Overview
# FastAPI test
# More tests - ping with curl

# Next Agent For Deployemnts / Cronjobs / Ingress
# How to Handle Secrets